# Reasoning-Based, Vectorless RAG with PageIndex and Vision-Language Models

### Libraries

Imports the `PageIndexClient` and utility helpers from the `pageindex` package, the async `AsyncGroq` client for calling the vision-language model, `PyMuPDF` (`fitz`) for PDF rendering, `base64` and `os` for file and encoding utilities, and the `PAGEINDEX_API_KEY` / `GROQ_API_KEY` credentials from `credentials.py`.

In [31]:
from pageindex import PageIndexClient
import pageindex.utils as utils
from credentials import PAGEINDEX_API_KEY, GROQ_API_KEY
from groq import AsyncGroq
import fitz, base64, os
from credentials import GROQ_API_KEY

### Setup PageIndex

Creates a `PageIndexClient` instance authenticated with the PageIndex API key. This client is used throughout the notebook to submit documents and retrieve their hierarchical tree structures.

In [2]:
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

### Setup VLM

Defines the async helper `call_vlm` that sends a text prompt (and optionally one or more JPEG images encoded as base64 data-URLs) to a Groq-hosted vision-language model. Images are read from local file paths, base64-encoded, and attached as `image_url` content parts. The function returns the model's trimmed text response.

In [3]:
async def call_vlm(prompt, image_paths=None, model="meta-llama/llama-4-scout-17b-16e-instruct"):
    client = AsyncGroq(api_key=GROQ_API_KEY)
    messages = [{"role": "user", "content": prompt}]
    if image_paths:
        content = [{"type": "text", "text": prompt}]
        for image in image_paths:
            if os.path.exists(image):
                with open(image, "rb") as image_file:
                    image_data = base64.b64encode(image_file.read()).decode('utf-8')
                    content.append({
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_data}"
                        }
                    })
        messages[0]["content"] = content
    response = await client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        max_completion_tokens=1024
    )
    return response.choices[0].message.content.strip()


#### PDF Image Extraction Helper Functions

Defines two helper functions:
- **`extract_pdf_page_images`** — opens a PDF with PyMuPDF, renders each page at 2× zoom for better quality, saves it as a JPEG under `pdf_images/`, and returns a `{page_number: image_path}` dict along with the total page count.
- **`get_page_images_for_nodes`** — given a list of node IDs and the node map, collects the image paths of all PDF pages spanned by those nodes, de-duplicating pages that are covered by multiple nodes.

In [4]:
def extract_pdf_page_images(pdf_path, output_dir="pdf_images"):
    os.makedirs(output_dir, exist_ok=True)
    pdf_document = fitz.open(pdf_path)
    page_images = {}
    total_pages = len(pdf_document)
    for page_number in range(len(pdf_document)):
        page = pdf_document.load_page(page_number)
        # Convert page to image
        mat = fitz.Matrix(2.0, 2.0)  # 2x zoom for better quality
        pix = page.get_pixmap(matrix=mat)
        img_data = pix.tobytes("jpeg")
        image_path = os.path.join(output_dir, f"page_{page_number + 1}.jpg")
        with open(image_path, "wb") as image_file:
            image_file.write(img_data)
        page_images[page_number + 1] = image_path
        print(f"Saved page {page_number + 1} image: {image_path}")
    pdf_document.close()
    return page_images, total_pages

def get_page_images_for_nodes(node_list, node_map, page_images):
    # Get PDF page images for retrieved nodes
    image_paths = []
    seen_pages = set()
    for node_id in node_list:
        node_info = node_map[node_id]
        for page_num in range(node_info['start_index'], node_info['end_index'] + 1):
            if page_num not in seen_pages:
                image_paths.append(page_images[page_num])
                seen_pages.add(page_num)
    return image_paths


## Step 1: Generate Tree Structure Using PageIndex

#### 1.1 Submit a document for generating PageIndex Tree Structure

Downloads the "Attention Is All You Need" PDF from arXiv and saves it to the `data/` directory. It then renders each PDF page to a JPEG image (used later as visual context for the VLM) and submits the PDF to PageIndex. The returned `doc_id` is used in all subsequent PageIndex API calls to reference this document.

In [27]:
import os, requests

pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"  # the "Attention Is All You Need" paper
pdf_path = os.path.join("../vectorless-rag/data", pdf_url.split('/')[-1])
os.makedirs(os.path.dirname(pdf_path), exist_ok=True)

response = requests.get(pdf_url)
with open(pdf_path, "wb") as f:
    f.write(response.content)
print(f"Downloaded {pdf_url}\n")

# Extract page images from PDF
print("Extracting page images...")
page_images, total_pages = extract_pdf_page_images(pdf_path)
print(f"Extracted {len(page_images)} page images from {total_pages} total pages.\n")

doc_id = pi_client.submit_document(pdf_path)["doc_id"]
print('Document Submitted:', doc_id)

Downloaded https://arxiv.org/pdf/1706.03762.pdf

Extracting page images...
Saved page 1 image: pdf_images\page_1.jpg
Saved page 2 image: pdf_images\page_2.jpg
Saved page 3 image: pdf_images\page_3.jpg
Saved page 4 image: pdf_images\page_4.jpg
Saved page 5 image: pdf_images\page_5.jpg
Saved page 6 image: pdf_images\page_6.jpg
Saved page 7 image: pdf_images\page_7.jpg
Saved page 8 image: pdf_images\page_8.jpg
Saved page 9 image: pdf_images\page_9.jpg
Saved page 10 image: pdf_images\page_10.jpg
Saved page 11 image: pdf_images\page_11.jpg
Saved page 12 image: pdf_images\page_12.jpg
Saved page 13 image: pdf_images\page_13.jpg
Saved page 14 image: pdf_images\page_14.jpg
Saved page 15 image: pdf_images\page_15.jpg
Extracted 15 page images from 15 total pages.

Document Submitted: pi-cmmoi42st00z5akqzl54bc4kq


#### 1.2 Get the generated tree from PageIndex

Checks whether PageIndex has finished processing the document. If ready, fetches the full hierarchical tree with per-node summaries and prints it in a human-readable format — excluding the raw text fields to keep the output concise. If not yet ready, it prompts the user to try again later.

In [11]:
if pi_client.is_retrieval_ready(doc_id):
    tree = pi_client.get_tree(doc_id, node_summary=True)['result']
    print('Simplified Tree Structure of the Document:')
    utils.print_tree(tree, exclude_fields=['text'])
else:
    print("Processing document, please try again later...")

Simplified Tree Structure of the Document:
[{'title': 'Attention Is All You Need',
  'node_id': '0000',
  'page_index': 1,
  'prefix_summary': '# Attention Is All You Need\n\nAshish Vasw...',
  'nodes': [{'title': 'Abstract',
             'node_id': '0001',
             'page_index': 1,
             'summary': 'The text introduces the Transformer, a n...'},
            {'title': '2 Background',
             'node_id': '0002',
             'page_index': 2,
             'summary': 'This text discusses the background of re...'},
            {'title': '3 Model Architecture',
             'node_id': '0003',
             'page_index': 2,
             'prefix_summary': 'The text describes the common encoder-de...',
             'nodes': [{'title': '3.1 Encoder and Decoder Stacks',
                        'node_id': '0004',
                        'page_index': 3,
                        'summary': 'The text describes the encoder and decod...'},
                       {'title': '3.2 Attention'

## Step 2: Reasoning-Based Retrieval with Tree Search

#### 2.1 Reasoning-based retrieval with PageIndex to identify nodes that might contain relevant context

Builds a structured prompt that presents the document tree (with summaries but without full text) alongside the user query. The VLM is asked to reason over the node titles and summaries to identify which nodes are most likely to contain the answer, and to return its reasoning and the relevant node IDs as a JSON object. `call_vlm` is then called with this prompt to get the result.

In [16]:
import json

query = "What is the last operation in the Scaled Dot-Product Attention figure?"

tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all tree nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = await call_vlm(search_prompt)

#### 2.2 Print retrieved nodes and VLM reasoning process

Parses the raw VLM response into a JSON object, stripping any Markdown code fences or extraneous text. Builds `node_map` — a dict mapping each node ID to its start/end page numbers. Then prints the model's chain-of-thought reasoning, followed by a table of retrieved nodes showing each node's ID, page range, and title.

In [17]:
import re
import json

def parse_tree_search_result(raw_text: str):
    text = raw_text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.IGNORECASE | re.DOTALL)

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        text = text[start:end + 1]

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        thinking_match = re.search(r'"thinking"\s*:\s*"(.*?)"\s*,\s*"node_list"', text, flags=re.DOTALL)
        node_list_match = re.search(r'"node_list"\s*:\s*\[(.*?)\]', text, flags=re.DOTALL)

        thinking = thinking_match.group(1).replace("\\n", "\n") if thinking_match else raw_text.strip()
        node_list = []
        if node_list_match:
            node_list = re.findall(r'"([^"\\]+)"', node_list_match.group(1))

        return {
            "thinking": thinking,
            "node_list": node_list
        }

node_map = utils.create_node_mapping(tree, include_page_ranges=True, max_page=total_pages)
tree_search_result_json = parse_tree_search_result(tree_search_result)

print('Reasoning Process:\n')
utils.print_wrapped(tree_search_result_json.get('thinking', ''))

print('\nRetrieved Nodes:\n')
for node_id in tree_search_result_json.get("node_list", []):
    if node_id not in node_map:
        continue
    node_info = node_map[node_id]
    node = node_info['node']
    start_page = node_info['start_index']
    end_page = node_info['end_index']
    page_range = start_page if start_page == end_page else f"{start_page}-{end_page}"
    print(f"Node ID: {node['node_id']}\t Pages: {page_range}\t Title: {node['title']}")

Reasoning Process:

To find the last operation in the Scaled Dot-Product Attention figure, we need to locate the node
that describes this figure and its operations. The Scaled Dot-Product Attention is mentioned in the
'3.2 Attention' section. Therefore, we should look into the nodes under this section to find the
specific details about the operations involved in Scaled Dot-Product Attention.

Retrieved Nodes:

Node ID: 0005	 Pages: 3-4	 Title: 3.2 Attention
Node ID: 0006	 Pages: 4	 Title: 3.2.1 Scaled Dot-Product Attention


#### 2.3 Get corresponding page images of retrieved nodes

Uses `get_page_images_for_nodes` to resolve the retrieved node IDs into a de-duplicated list of JPEG page image paths. These images represent the exact PDF pages that contain the relevant content and will be passed directly to the VLM as visual grounding context for answer generation.

In [18]:
retrieved_nodes = tree_search_result_json["node_list"]
retrieved_page_images = get_page_images_for_nodes(retrieved_nodes, node_map, page_images)
print(f'\nRetrieved {len(retrieved_page_images)} PDF page image(s) for visual context.')


Retrieved 2 PDF page image(s) for visual context.


Constructs a visual QA prompt instructing the VLM to answer the query solely from the retrieved PDF page images. Calls `call_vlm` with both the prompt and the image paths so the model can visually inspect the relevant pages. The model is explicitly told to state "The answer is not found in the provided document." if the evidence is absent.

In [26]:
# Generate answer using VLM with only PDF page images as visual context
answer_prompt = f"""
Answer the question based on the images of the document pages as context.

Question: {query}

Provide a clear answer based only on the context provided. Give your thinking process if possible. Make sure your answer is correct. If the answer cannot be found in the provided context, say "The answer is not found in the provided document."
"""

print('Generated answer using VLM with retrieved PDF page images as visual context:\n')
answer = await call_vlm(answer_prompt, retrieved_page_images)
utils.print_wrapped(answer)

Generated answer using VLM with retrieved PDF page images as visual context:

To determine the last operation in the Scaled Dot-Product Attention figure, let's analyze the
provided image of the document pages, specifically focusing on Figure 2, which illustrates the
Scaled Dot-Product Attention.

The Scaled Dot-Product Attention figure shows a series of operations performed in sequence. The
steps are as follows:

1. **MatMul (Q, K)**: The first operation involves a matrix multiplication between Query (Q) and Key
(K).
2. **Scale**: The result of the matrix multiplication is then scaled.
3. **Mask (opt.)**: Optionally, a mask is applied.
4. **Softmax**: The scaled result (with or without masking) is then passed through a softmax
function.
5. **MatMul**: Finally, the output of the softmax function is matrix multiplied by Value (V).

Based on this sequence of operations, the last operation in the Scaled Dot-Product Attention figure
is the **MatMul** operation, where the output of the softm